---
# Praxisteil 4 - Metaheuristiken
---

---
## 1. Iterated Greedy
---

---
### 1.1 Wiederholung allgemeiner Ablauf
---

Ablauf Iterated Greedy

![AblaufIG](../DataFiles/IteratedGreedyAblauf.png)

----
### 1.2 Startlösung
----
NEH-Heuristik

In [77]:
from Solver import *

data = InputData("../TestInstancesJson/Small/VFR10_5_5_SIST.json")

solver = Solver(data, 1060) #1014, 754 |  1060, 736

startSolution = solver.ConstructionPhase("NEH")

currentSolution = deepcopy(startSolution)


Generating an initial solution according to NEH.
Constructive solution found.
The permutation [4, 3, 7, 1, 8, 2, 9, 0, 5, 6] results in a Makespan of 752


----
### 1.3 Destruktion
----
Entferne zufällig _numberJobsToRemove_ Aufträge aus aktueller Lösung. Dabei ist _numberJobsToRemove_ ein Parameter des Iterated Greedy.
Als Ergebnis ensteht die Menge entfernter Aufträge _removedJobs_ und eine unvollständige Lösung _partialPermutation_.

In [78]:
# Eigene Lösung
numberJobsToRemove = 2 # 1st parameter

import numpy as np
def removeJobsRandomly(currentSolution, numberJobsToRemove):
    jobstoRemove = np.random.choice(range(len(currentSolution.Permutation)), numberJobsToRemove, replace=False)
    removedSolution = deepcopy(currentSolution)
    removedSolution.Permutation = [job for i, job in enumerate(removedSolution.Permutation) if i not in jobstoRemove]
    return removedSolution, jobstoRemove

removedSolution, jobstoRemove = removeJobsRandomly(currentSolution, numberJobsToRemove)
print("Jobs removed:", jobstoRemove)
print("Removed solution:", removedSolution.Permutation)

Jobs removed: [7 2]
Removed solution: [4, 3, 1, 8, 2, 9, 5, 6]


In [79]:
# Musterlösung
numberJobsToRemove = 2 # 1st parameter

def Destruction(solution):
        #https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html
        removedJobs = solver.RNG.choice(solver.InputData.n, size=numberJobsToRemove, replace = False).tolist()

        partialPermutation = [i for i in solution.Permutation if i not in removedJobs]

        return removedJobs, partialPermutation

removedJobs, partialPermutation = Destruction(currentSolution)

print(f'Removed jobs: {removedJobs}')
print(f'Partial permutation: {partialPermutation}')

Removed jobs: [0, 9]
Partial permutation: [4, 3, 7, 1, 8, 2, 5, 6]


----
### 1.4 Konstruktion
----
Füge die _removedJobs_ unter Beachtung der Reihenfolge an der jeweils besten Position (NEH) wieder zu _partialPermutation_ hinzu und gib die neue vollständige Lösung zurück. 

In [80]:
def construction(partialPermutation, removedJobs):
    completeSolution = Solution(solver.InputData.InputJobs, partialPermutation)
    for job in removedJobs:
        solver.EvaluationLogic.DetermineBestInsertion(completeSolution, job)
    return completeSolution

newSolution = construction(partialPermutation, removedJobs)
print(f'New solution: {newSolution.Permutation}, makespan: {newSolution.Makespan}')

New solution: [4, 3, 0, 7, 1, 8, 2, 9, 5, 6], makespan: 736


----
### 1.5 Akzeptanz und Gedächtnis
---- 
Akzeptiere neue Lösung als aktuelle Lösung, wenn
* neue Lösung besser als aktuelle Lösung (immer)
* Akzeptanzkriterium erfüllt (zufällig)

Akzeptanzkriterium bei Ruiz/ Stützle:  $\pi''$ - neue Lösung (_newSolution_), $\pi$ - aktuelle Lösung (_currentSolution_)
$$
    \begin{equation}
    e^{- \frac{C_{\max}\left( \pi'' \right) - C_{\max}\left( \pi \right)}{Temperature}} \text{ mit } Temperature = T \cdot \frac{\sum Bearbeitungszeiten}{n \cdot m \cdot 10}
    \end{equation}
$$  
$T$ ist der 2. Parameter des Iterated Greedy. Je höher $T$ ist, desto größer ist die Wahrscheinlichkeit, schlechtere Lösungen zu akzeptieren.

Neue beste Lösung wird im _SolutionPool_ gespeichert.

In [71]:
len(solver.InputData.InputJobs[1].Operations())

5

In [ ]:
print(f'Solution before checking for acceptance: \n{currentSolution}')

baseTemperature = 1 # 2nd parameter
def AkzeptanzKriterium(currentSolution, newSolution, baseTemperature, iteration = 1):
    if newSolution.Makespan < currentSolution.Makespan:
        currentSolution = newSolution

        if newSolution.Makespan < solver.SolutionPool.GetLowestMakespanSolution().Makespan:
            print(f'New best solution found in iteration {iteration}:\n{newSolution}')
            solver.SolutionPool.AddSolution(newSolution)
    
    else:
        totalprocessingTime = sum(x.ProcessingTimes[i] for x in solver.InputData.InputJobs for i in range(len(x.Operations())))
        temperature = baseTemperature*(totalprocessingTime / (solver.InputData.n * solver.InputData.m * 10))
        probability = math.exp(-(currentSolution.Makespan - newSolution.Makespan) / temperature)
        randomValue = solver.RNG.random()
        if randomValue < probability:
            currentSolution = newSolution
    return currentSolution

currentSolution = AkzeptanzKriterium(currentSolution, newSolution, baseTemperature)
print(f'Solution after checking for acceptance: \n{currentSolution}')

Solution before checking for acceptance: 
The permutation [4, 3, 7, 1, 8, 2, 9, 0, 5, 6] results in a Makespan of 752
New best solution found in iteration 1:
The permutation [4, 3, 0, 7, 1, 8, 2, 9, 5, 6] results in a Makespan of 736
Solution after checking for acceptance: 
The permutation [4, 3, 0, 7, 1, 8, 2, 9, 5, 6] results in a Makespan of 736


Akzeptanzkriterium in Funktion für bessere Übersichtlichkeit

In [82]:
def AcceptWorseSolution(currentObjectiveValue, newObjectiveValue):  
    temperature = baseTemperature * solver.InputData.TotalProcessingTime / (solver.InputData.n * solver.InputData.m * 10)
    probability = math.exp(-(newObjectiveValue - currentObjectiveValue) / temperature)

    return solver.RNG.random() <= probability

----
### 1.6 Lokale Suche
----
Verfahren der lokalen Suche können optional verwendet werden, um die aktuelle Lösung zu verbessern. Häufig wird __Iterative Improvement__ mit der Insertion-Nachbarschaft verwendet.

In [74]:
print(f'Solution before Local Search:\n{currentSolution}')

improvedSolution = IterativeImprovement(solver.InputData, 'BestImprovement', ['Insertion'])
improvedSolution.Initialize(solver.EvaluationLogic, solver.SolutionPool)
currentSolution = improvedSolution.Run(currentSolution)
print(f'Solution after nach Local Search\n{currentSolution}')

Solution before Local Search:
The permutation [4, 3, 0, 7, 1, 8, 2, 9, 5, 6] results in a Makespan of 736
Solution after nach Local Search
The permutation [1, 8, 4, 3, 0, 5, 2, 7, 9, 6] results in a Makespan of 713


----
### 1.7 Iterated Greedy als Steuerungsmechanismus der Komponenten
----
Iterated Greedy ist iterative Abfolge von Destruktion und Konstruktion $\Rightarrow$ wird in Schleife immer wieder durchgeführt bis Stoppkriterium erreicht ist. 

Hier im Beispiel ist Stoppkriterium die Anzahl an Iterationen _maxIterations_, aber bspw. auch Zeitlimit oder Anzahl Iterationen ohne Verbesserungen möglich.

In [83]:
maxIterations = 10 # stop criterion number of iterations

def Run(currentSolution, maxIterations):
    localSearch = IterativeImprovement(solver.InputData, 'BestImprovement', ['Insertion'])
    localSearch.Initialize(solver.EvaluationLogic, solver.SolutionPool)

    currentSolution = localSearch.Run(currentSolution)
    currentSolution = solver.SolutionPool.GetLowestMakespanSolution()
    iteration = 0
    while iteration < maxIterations:
        removedJobs, partialPermutation = Destruction(currentSolution)
        newSolution = construction(partialPermutation, removedJobs)
        newSolution = localSearch.Run(newSolution)
        currentSolution = AkzeptanzKriterium(currentSolution, newSolution, baseTemperature, iteration)
        iteration += 1
    return currentSolution

iteratedGreedySol = Run(currentSolution, maxIterations)
print(f'Solution after Iterated Greedy:\n{iteratedGreedySol}')

Solution after Iterated Greedy:
The permutation [1, 8, 3, 4, 0, 5, 2, 9, 7, 6] results in a Makespan of 713


Iterated Greedy ausführen

In [84]:
solver = Solver(data, 1060)

startSolution = solver.ConstructionPhase("NEH")

bestSolution = Run(startSolution, maxIterations)

print(f'Best solution found.\n{bestSolution}')

Generating an initial solution according to NEH.
Constructive solution found.
The permutation [4, 3, 7, 1, 8, 2, 9, 0, 5, 6] results in a Makespan of 752
Best solution found.
The permutation [1, 8, 3, 4, 0, 5, 2, 9, 7, 6] results in a Makespan of 713


---
## 2. Klasse für Iterated Greedy
---

Analog wie für den Algorithmus _IterativeImprovement_ soll jetzt auch für _IteratedGreedy_ eine eigene Klasse angelegt werden, damit eine Instanz der Klasse an den Solver übergeben werden kann.

_Iterated Greedy_ soll wie _IterativeImprovement_ von _ImprovementAlgorithm_ erben.

Notwendige __Parameter__ sind Attribute:
* NumberJobsToRemove
* BaseTemperature
* MaxIterations
* Instanz des Local Search Algorithmus

EvaluationLogic, SolutionPool und random number generator werden von Solver an Algorithmus übergeben.

__Programmieraufgabe:__ Ergänzen sie die Klasse _IteratedGreedy_, sodass sie an den Solver als Algorithmus übergeben werden kann.

__Funktionen:__ Konstruktor, Initialize, Destruction, Construction, AcceptWorseSolution, Run

In [95]:
class IteratedGreedy(ImprovementAlgorithm):
    def __init__(self, inputData, numberJobsToRemove, baseTemperature, maxIterations, localSearchAlgorithm=None):
        super().__init__(inputData)
        self.numberJobsToRemove = numberJobsToRemove
        self.baseTemperature = baseTemperature
        self.maxIterations = maxIterations
        self.localSearchAlgorithm = localSearchAlgorithm if localSearchAlgorithm else IterativeImprovement(self.inputData, 'BestImprovement', neighborhoodTypes=[])   # ohne gewählte Nachbarschaft wird kein LS ausgeführt

    def Initialize(self, evaluationLogic, solutionPool, rng):
        super().Initialize(evaluationLogic, solutionPool, rng)
        self.localSearchAlgorithm.Initialize(self.EvaluationLogic, self.SolutionPool)

    def Destruction(self, currentSolution):
        removedJobs = self.RNG.choice(self.InputData.n, size=self.numberJobsToRemove, replace=False).tolist()
        partialPermutation = [i for i in currentSolution.Permutation if i not in removedJobs]
        return removedJobs, partialPermutation
    
    def Construction(self, partialPermutation, removedJobs):
        completeSolution = Solution(self.InputData.InputJobs, partialPermutation)
        for job in removedJobs:
            self.EvaluationLogic.DetermineBestInsertion(completeSolution, job)
        return completeSolution
    
    def AcceptWorseSolution(self, currentSolution, newSolution):
        totalProcessingTime = sum(x.ProcessingTimes[i] for x in self.InputData.InputJobs for i in range(len(x.Operations())))
        # alternativ auch direkt self.InputData.TotalProcessingTime möglich, wenn zuvor in InputData definiert
        temperature = self.baseTemperature * (totalProcessingTime / self.InputData.n * self.InputData.m * 10)
        probability = math.exp(-(newSolution.Makespan - currentSolution.Makespan) / temperature)
        return self.RNG.random() <= probability
    
    def Run(self, currentSolution):
        currentSolution = self.localSearchAlgorithm.Run(currentSolution)

        currentBest = self.SolutionPool.GetLowestMakespanSolution().Makespan
        iteration = 0
        while iteration < self.maxIterations:
            removedJobs, partialPermutation = self.Destruction(currentSolution)
            newSolution = self.Construction(partialPermutation, removedJobs)
            newSolution = self.localSearchAlgorithm.Run(newSolution)
            if newSolution.Makespan < currentSolution.Makespan:
                currentSolution = newSolution

                if newSolution.Makespan < currentBest:
                    print(f'New best Solution found in iteration {iteration}:\n{newSolution}')
                    self.SolutionPool.AddSolution(newSolution)
                    currentBest = newSolution.Makespan

            elif self.AcceptWorseSolution(currentSolution, newSolution):
                    currentSolution = newSolution

            iteration += 1
        return self.SolutionPool.GetLowestMakespanSolution()
    

Der Algorithmus kann jetzt in _RunLocalSearch()_ an den Solver übergeben und ausgeführt werden.

In [104]:
data = InputData("../TestInstancesJson/Small/VFR40_10_3_SIST.json") #"../TestInstancesJson/Small/VFR40_10_3_SIST.json"

insertion = IterativeImprovement(data, 'BestImprovement', ['Insertion'])
taillardInsertion = IterativeImprovement(data, 'BestImprovement', ['TaillardInsertion'])
blockMoveK3 = IterativeImprovement(data, 'BestImprovement', ['BlockK3'])
swapMove = IterativeImprovement(data, 'BestImprovement', ['Swap'])
iteratedGreedy = IteratedGreedy(
    data, 
    numberJobsToRemove=2, 
    baseTemperature=1, 
    maxIterations=100,
    localSearchAlgorithm=taillardInsertion)

solver = Solver(data, 1010)

solver.RunLocalSearch(
    constructiveSolutionMethod='NEH',
    algorithm=iteratedGreedy)

Generating an initial solution according to NEH.
Constructive solution found.
The permutation [31, 24, 15, 33, 4, 2, 16, 1, 13, 39, 36, 11, 37, 20, 7, 14, 29, 26, 28, 23, 27, 19, 32, 10, 35, 38, 25, 8, 12, 30, 5, 22, 6, 18, 9, 3, 34, 17, 0, 21] results in a Makespan of 2545
New best Solution found in iteration 0:
The permutation [20, 24, 19, 15, 33, 4, 2, 17, 16, 1, 13, 39, 36, 11, 37, 7, 14, 26, 28, 23, 3, 29, 32, 0, 10, 35, 38, 25, 8, 12, 30, 27, 31, 5, 22, 6, 18, 9, 34, 21] results in a Makespan of 2476
New best Solution found in iteration 3:
The permutation [20, 39, 24, 19, 15, 33, 4, 2, 17, 1, 13, 36, 37, 7, 14, 26, 11, 28, 23, 16, 29, 3, 32, 0, 10, 35, 38, 25, 8, 12, 30, 27, 31, 5, 22, 18, 9, 6, 34, 21] results in a Makespan of 2450
New best Solution found in iteration 40:
The permutation [20, 39, 24, 28, 19, 15, 36, 23, 37, 32, 33, 4, 17, 2, 1, 7, 14, 35, 26, 0, 16, 25, 3, 10, 8, 12, 30, 29, 31, 38, 22, 27, 11, 6, 13, 18, 9, 5, 34, 21] results in a Makespan of 2441
New best Solu

----
## 3. Exkurs: DEAP und Genetic Algorithms
----

Distributed Evolutionary Algorithms in Python ([DEAP](https://github.com/DEAP/deap)) ist ein Framework für Evolutionary Computation, mit u.a. den populationsbasierten Metaheuristiken Genetic Algorithms und Particle Swarm Optimization.

![Metaheuristiken](../DataFiles/MH.png)

### Wichtige Begriffe für Genetic Algorithms:
* Individuum: Einzelne Lösung
* Population: Sammlung von Individuen
* Generation: Population in einer Iteration
* Reproduktion: Erzeugung neuer Individuen aus bekannten Individuen
* Selektion: Auswahl von Individuen für Reproduktion und neue Generation
* Fitness: Bewertung eines Individuums (Zielfunktionswert)
* Mutation: Zufällige Veränderung eines Individuums

![GeneticAlgorithm](../DataFiles/GA.png)

----
### Beispiel DEAP
----

__Creator__

Meta-Factory um Klassen zu erzeugen

Parameters:
* name – The name of the class to create.
* base – A base class from which to inherit.
* attribute – One or more attributes to add on instantiation of this class, optional.


In [105]:
from Solver import *

from deap import algorithms
from deap import base
from deap import creator
from deap import tools

data = InputData("../TestInstancesJson/Small/VFR10_5_5_SIST.json")

solver = Solver(data, 1011)

creator.create("FitnessMin", base.Fitness, weights=[-1.0]) # minimization problem, weights as list
creator.create("Individual", list, fitness=creator.FitnessMin) # single individual stored in python list

__Toolbox__

Erzeugt Konfiguration des Algorithmus

In [106]:
toolbox = base.Toolbox() 

# Individual is a permutation of integer indices
toolbox.register("indices", solver.RNG.choice, range(solver.InputData.n), solver.InputData.n, replace=False)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)

# Population is a collection of individuals
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
pop = toolbox.population(100) # 100 individuals in population

# Operators
toolbox.register("mate", tools.cxPartialyMatched) # set crossover 
toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.05) # set mutation
toolbox.register("select", tools.selTournament, tournsize=3) # set selection mechanism

__Fitness-Funktion__

Bewertung eines Indivduums

In [107]:
def EvalPFSP(solver, individual):
    solution = Solution(solver.InputData.InputJobs, individual)
    solver.EvaluationLogic.DefineStartEnd(solution)
    return [solution.Makespan]

toolbox.register("evaluate", EvalPFSP, solver) # fitness function

__Statistiken__

Sammlung von Werten während Laufzeit des Algorithmus'

In [108]:
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", numpy.mean)
stats.register("std", numpy.std)
stats.register("min", numpy.min)
stats.register("max", numpy.max)

__Hall of fame__

Gedächtnis zum Speichern der besten Lösungen; hier nur beste Lösung

In [109]:
hof = tools.HallOfFame(1)

__Ausführen__

In [111]:
random.seed(solver.Seed)
algorithms.eaSimple(population=pop, toolbox=toolbox, cxpb=0.8, mutpb=0.2, ngen=100, stats=stats, halloffame=hof)

bestPermutation = hof[0]
bestSolution = Solution(solver.InputData.InputJobs, bestPermutation)
solver.EvaluationLogic.DefineStartEnd(bestSolution)
print(f'Best found Solution.\n {bestSolution}')

gen	nevals	avg   	std    	min	max
0  	0     	729.12	13.9766	726	824
1  	81    	728.84	13.2648	726	822
2  	82    	729.07	12.9261	726	813
3  	91    	727.21	10.5521	726	831
4  	81    	727.34	8.34652	726	792
5  	78    	728.2 	11.8701	726	804
6  	85    	727.62	8.03216	726	772
7  	76    	729.31	15.1609	726	814
8  	90    	728.92	10.0684	726	792
9  	84    	729.96	17.9621	726	840
10 	84    	729.82	16.1105	726	824
11 	83    	728.07	12.6746	726	835
12 	89    	728.68	12.0274	726	804
13 	77    	730.91	20.1703	726	840
14 	90    	728.39	11.0814	726	816
15 	80    	730.3 	18.8369	726	831
16 	88    	731.21	18.2693	726	825
17 	82    	727.2 	6.37181	726	771
18 	90    	731.91	18.4662	726	824
19 	77    	727.97	9.27195	726	788
20 	90    	727.31	6.56764	726	768
21 	84    	730.32	17.3775	726	831
22 	81    	732.8 	20.8384	726	841
23 	80    	727.79	8.2877 	726	775
24 	90    	728.46	13.4539	726	840
25 	86    	728.8 	14.3618	726	840
26 	87    	727.99	11.0856	726	816
27 	82    	730.93	15.7545	726	814
28 	77    	729

----
#### Funktion für Solver
----

Parameter als Argumente
* Populationsgröße _populationSize_
* Anzahl Generationen _generations_
* Rekombinationswahrscheinlichkeit _matingProb_
* Mutationswahrscheinlichkeit _mutationProb_

In [112]:
def EvalPFSP(self, individual):
    solution = Solution(self.InputData.InputJobs, individual)
    self.EvaluationLogic.DefineStartEnd(solution)
    return [solution.Makespan]

def RunGeneticAlgorithm(self, populationSize, generations, matingProb, mutationProb):
    # Creator - meta-factory to create new classes
    creator.create("FitnessMin", base.Fitness, weights=[-1.0])
    creator.create("Individual", list, fitness=creator.FitnessMin)

    toolbox = base.Toolbox() 

    # Individual is a permutation of integer indices
    toolbox.register("indices", solver.RNG.choice, range(self.InputData.n), self.InputData.n, replace=False)
    toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)

    # Population is a collection of individuals
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    pop = toolbox.population(populationSize) # number of individuals in population

    # Operators
    toolbox.register("mate", tools.cxPartialyMatched) # set crossover 
    toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.05) # set mutation
    toolbox.register("select", tools.selTournament, tournsize=3) # set selection mechanism
    
    # Fitness function
    toolbox.register("evaluate", self.EvalPFSP) 
    
    # Statistics during run time
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", numpy.mean)
    stats.register("std", numpy.std)
    stats.register("min", numpy.min)
    stats.register("max", numpy.max)
            
    # Hall of fame --> Best individual
    hof = tools.HallOfFame(1)

    random.seed(self.Seed)
    algorithms.eaSimple(population=pop, toolbox=toolbox, cxpb=matingProb, mutpb=mutationProb, ngen=generations, stats=stats, halloffame=hof)

    bestPermutation = hof[0]
    bestSolution = Solution(self.InputData.InputJobs, bestPermutation)
    self.EvaluationLogic.DefineStartEnd(bestSolution)
    print(f'Best found Solution.\n {bestSolution}')

Solver.EvalPFSP = EvalPFSP
Solver.RunGeneticAlgorithm = RunGeneticAlgorithm

Vergleich Iterated Greedy und Genetic Algorithm

In [113]:
data = InputData("../TestInstancesJson/Large/VFR100_20_1_SIST.json")
taillardInsertion = IterativeImprovement(data, 'FirstImprovement', neighborhoodTypes=['TaillardInsertion'])
iteratedGreedy = IteratedGreedy(
    data, 
    numberJobsToRemove=2, 
    baseTemperature=1, 
    maxIterations=50, 
    localSearchAlgorithm=taillardInsertion)

solver = Solver(data, 1010)

In [114]:
solver.RunLocalSearch(
    constructiveSolutionMethod='NEH',
    algorithm=iteratedGreedy)

Generating an initial solution according to NEH.
Constructive solution found.
The permutation [61, 80, 51, 99, 11, 8, 62, 41, 24, 94, 67, 63, 0, 27, 31, 30, 82, 52, 91, 1, 19, 59, 89, 4, 49, 88, 25, 47, 85, 81, 38, 23, 5, 56, 71, 93, 83, 2, 64, 33, 96, 20, 21, 77, 78, 13, 60, 74, 98, 90, 6, 76, 9, 3, 42, 84, 50, 86, 95, 72, 97, 35, 40, 92, 15, 39, 10, 68, 43, 17, 26, 36, 18, 57, 44, 29, 54, 79, 55, 22, 48, 53, 75, 16, 73, 12, 58, 28, 32, 87, 70, 34, 7, 66, 45, 69, 14, 65, 46, 37] results in a Makespan of 6596
New best solution in iteration 2: The permutation [51, 61, 28, 80, 8, 62, 99, 31, 52, 63, 11, 94, 41, 67, 1, 82, 30, 91, 24, 4, 59, 47, 49, 88, 81, 85, 25, 38, 27, 23, 89, 71, 5, 56, 93, 83, 33, 96, 20, 77, 64, 17, 13, 19, 78, 60, 98, 90, 21, 6, 76, 43, 9, 3, 84, 86, 95, 40, 10, 35, 42, 74, 92, 15, 39, 97, 68, 26, 36, 18, 57, 0, 44, 29, 50, 54, 79, 55, 22, 48, 53, 75, 16, 73, 12, 58, 32, 87, 70, 34, 7, 66, 72, 45, 69, 14, 2, 65, 46, 37] results in a Makespan of 6424
New best solut

In [120]:
solver.RunGeneticAlgorithm(
    populationSize=100, 
    generations=200, 
    matingProb=0.8, 
    mutationProb=0.2)


gen	nevals	avg    	std    	min 	max 
0  	100   	7828.65	144.613	7470	8136
1  	78    	7758.42	127.643	7470	8110
2  	78    	7728.65	127.53 	7470	8123
3  	76    	7738.61	145.115	7462	8081
4  	77    	7699.89	138.737	7439	8027
5  	90    	7681.7 	144.452	7421	8006
6  	78    	7623.82	138.748	7388	7959
7  	86    	7610.36	148.108	7391	8085
8  	82    	7588.62	138.76 	7324	7970
9  	76    	7546.97	109.282	7324	7880
10 	81    	7547.75	126.981	7324	7913
11 	88    	7528.95	124.491	7309	7937
12 	87    	7501.09	106.884	7298	7773
13 	86    	7478.56	120.764	7280	7801
14 	87    	7475.69	136.756	7189	7870
15 	86    	7443.83	125.546	7160	7825
16 	85    	7452.05	134.236	7160	7917
17 	71    	7417.7 	105.477	7237	7815
18 	91    	7422.56	105.374	7192	7681
19 	85    	7443.48	124.823	7192	7891
20 	89    	7422.5 	125.585	7192	7803
21 	86    	7403.47	120.389	7192	7769
22 	92    	7406.62	133.178	7116	7746
23 	84    	7387.96	137.82 	7116	7819
24 	93    	7383.14	108.673	7155	7693
25 	83    	7353.68	111.463	7155	7677
2